# AI Assistant: Order Up Decision Analysis

This notebook uses the AI Assistant to analyze order up decisions.

Example: If King of Hearts is turned up and you have both bowers and the ace, the AI will show "90% chance of winning hand".


In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from eucher.ai_players.ai_assistant import AIAssistant
from eucher.cards import Card, Deck, Rank, Suit
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets


In [ ]:
# Initialize AI Assistant
thinking_time_dropdown = widgets.Dropdown(
    options=['fast', 'quick', 'normal', 'thorough', 'deep'],
    value='normal',
    description='Thinking Time:',
    style={'description_width': 'initial'}
)

assistant = AIAssistant(thinking_time=thinking_time_dropdown.value)

def update_thinking_time(change):
    global assistant
    assistant = AIAssistant(thinking_time=change['new'])

thinking_time_dropdown.observe(update_thinking_time, names='value')

display(thinking_time_dropdown)


In [ ]:
# Example: Analyze order up decision
deck = Deck()
deck.shuffle()
hand = deck.deal(5)
turned_card = deck.draw_one()

# Example: Strong hand (both bowers + ace)
# In practice, you would set your actual hand here
example_hand = [
    Card(Suit.HEARTS, Rank.JACK),  # Right bower (if hearts is trump)
    Card(Suit.DIAMONDS, Rank.JACK),  # Left bower (if hearts is trump)
    Card(Suit.HEARTS, Rank.ACE),
    Card(Suit.HEARTS, Rank.KING),
    Card(Suit.HEARTS, Rank.QUEEN),
]

example_turned = Card(Suit.HEARTS, Rank.TEN)

display(HTML(f"""
<div style="border: 2px solid #333; padding: 10px; margin: 10px;">
    <h3>Order Up Decision Analysis</h3>
    <p><strong>Your Hand:</strong> {', '.join(str(c) for c in example_hand)}</p>
    <p><strong>Turned Card:</strong> {example_turned}</p>
</div>
"""))

analyze_button = widgets.Button(description="Analyze Order Up Decision", button_style='success')
result_output = widgets.Output()

def analyze_order_up(b):
    with result_output:
        clear_output()
        display(HTML("<p>Running Monte Carlo simulation... This may take a moment.</p>"))
        
        result = assistant.analyze_order_up_decision(
            hand=example_hand,
            turned_card=example_turned,
            dealer_id=3,
            player_id=0,
            team=0
        )
        
        win_prob = result['win_probability']
        confidence = result['confidence']
        simulations = result['simulations']
        
        # Color code based on probability
        color = 'green' if win_prob >= 0.7 else 'orange' if win_prob >= 0.5 else 'red'
        
        clear_output()
        display(HTML(f"""
        <div style="border: 2px solid #333; padding: 10px; margin: 10px;">
            <h3>Analysis Results</h3>
            <p style="font-size: 24px; color: {color};">
                <strong>{win_prob*100:.1f}% chance of winning hand</strong>
            </p>
            <p><strong>Confidence:</strong> {confidence*100:.1f}%</p>
            <p><strong>Simulations:</strong> {simulations}</p>
            <p><strong>Recommendation:</strong> {'Order Up' if win_prob >= 0.5 else 'Pass'}</p>
        </div>
        """))

analyze_button.on_click(analyze_order_up)
display(analyze_button)
display(result_output)
